# W3 DOM feature notebook
Vũ Văn Bình — snapshot/parser v0.1. Phân tích 17 trạng thái M0 đã thu; chưa có candidate generator hoặc số đo recall.


In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT = Path.cwd()
if not (ROOT / 'dom_snapshot').exists():
    ROOT = ROOT.parent
SAMPLES = ROOT / 'results/w3/dom_samples_expanded_v0.1'
records = [json.loads(p.read_text(encoding='utf-8')) for p in sorted(SAMPLES.glob('*/features.json'))]
assert len(records) == 17, 'Expected seventeen sample states'
for r in records:
    print(r['state'], 'elements=', len(r['elements']))


customers-detail elements= 32
customers-edit elements= 24
customers-list elements= 58
customers-new elements= 29
customers-search elements= 40
orders-detail elements= 42
orders-filtered elements= 56
orders-list elements= 68
orders-new elements= 41
orders-two-lines elements= 51
orders-updated elements= 42
tickets-assigned elements= 37
tickets-closed elements= 25
tickets-detail elements= 37
tickets-list elements= 41
tickets-new elements= 31
tickets-prefilled elements= 31


## Đặc trưng phần tử
Control ở đây là input/select/textarea/button/a, chưa phải danh sách ứng viên đã lọc. Visible không đồng nghĩa với có thể click.


In [2]:
for r in records:
    elements = r['elements']
    controls = [e for e in elements if e['tag'] in ('input','select','textarea','button','a')]
    labels = sum(e['label'] is not None for e in controls)
    print(r['state'], 'tags=', dict(Counter(e['tag'] for e in elements)),
          'controls=', len(controls), 'labeled_controls=', labels,
          'visible_controls=', sum(e['visible'] for e in controls))


customers-detail tags= {'html': 1, 'head': 1, 'meta': 2, 'title': 1, 'link': 1, 'body': 1, 'header': 1, 'div': 1, 'a': 7, 'span': 1, 'nav': 1, 'main': 1, 'h1': 1, 'p': 3, 'dl': 1, 'dt': 4, 'dd': 4} controls= 7 labeled_controls= 0 visible_controls= 7
customers-edit tags= {'html': 1, 'head': 1, 'meta': 2, 'title': 1, 'link': 1, 'body': 1, 'header': 1, 'div': 1, 'a': 5, 'span': 1, 'nav': 1, 'main': 1, 'h1': 1, 'p': 2, 'form': 1, 'label': 1, 'input': 1, 'button': 1} controls= 7 labeled_controls= 1 visible_controls= 7
customers-list tags= {'html': 1, 'head': 1, 'meta': 2, 'title': 1, 'link': 1, 'body': 1, 'header': 1, 'div': 1, 'a': 11, 'span': 1, 'nav': 1, 'main': 1, 'h1': 1, 'form': 1, 'label': 1, 'input': 1, 'button': 1, 'p': 2, 'table': 1, 'thead': 1, 'tr': 5, 'th': 4, 'tbody': 1, 'td': 16} controls= 13 labeled_controls= 1 visible_controls= 13
customers-new tags= {'html': 1, 'head': 1, 'meta': 2, 'title': 1, 'link': 1, 'body': 1, 'header': 1, 'div': 1, 'a': 5, 'span': 1, 'nav': 1, 'main

## Kiểm tra trạng thái động
Thêm dòng sản phẩm phải xuất hiện select và input mới, kèm label liên kết đúng.


In [3]:
by_state = {r['state']: r for r in records}
before = by_state['orders-new']['elements']
after = by_state['orders-two-lines']['elements']
added = {e['attributes'].get('id'): e for e in after}
assert added['order-sku-2']['label'] == 'Sản phẩm'
assert added['order-qty-2']['label'] == 'Số lượng'
assert added['order-sku-2']['visible'] and added['order-qty-2']['visible']
print('Element increase:', len(after) - len(before))
print('New labels:', added['order-sku-2']['label'], '/', added['order-qty-2']['label'])


Element increase: 10
New labels: Sản phẩm / Số lượng


## Tính toàn vẹn và bàn giao
Kiểm tra hash HTML và schema trước khi Trung đọc JSON. ID/XPath chỉ có hiệu lực trong snapshot tương ứng.


In [4]:
import sys, hashlib
sys.path.insert(0, str(ROOT))
from dom_snapshot.parser import validate_snapshot
for r in records:
    validate_snapshot(r)
    html = (SAMPLES / r['snapshot_id'] / 'dom.html').read_bytes()
    assert hashlib.sha256(html).hexdigest() == r['html_sha256']
print('PASS: 17 schemas and 17 HTML checksums')


PASS: 17 schemas and 17 HTML checksums


In [5]:
coverage = json.loads((SAMPLES / 'coverage.json').read_text(encoding='utf-8'))
assert set(coverage['tasks']) == {f'T{i:03}' for i in range(1,21)}
for task, states in coverage['tasks'].items():
    print(task, ':', ', '.join(states))
for state, control, value in [('customers-edit','customer-phone','0900000001'), ('orders-updated','order-status','delivered'), ('tickets-assigned','ticket-assignee','1')]:
    element = next(e for e in by_state[state]['elements'] if e['attributes'].get('id') == control)
    assert element['value'] == value
closed = by_state['tickets-closed']['elements']
assert not any(e['attributes'].get('id') in ('ticket-assignee','ticket-note','close-ticket') for e in closed)
print('PASS: coverage map and updated runtime values')


T001 : customers-list, customers-new
T002 : customers-list, customers-search, customers-detail
T003 : customers-list, customers-search, customers-detail, customers-edit
T004 : orders-list, orders-new, orders-two-lines
T005 : orders-list, orders-detail, orders-updated
T006 : orders-list, orders-filtered
T007 : tickets-list, tickets-new
T008 : tickets-list, tickets-detail, tickets-assigned
T009 : tickets-list, tickets-detail, tickets-closed
T010 : customers-list, customers-search, customers-detail, orders-new, orders-detail, tickets-new, tickets-prefilled
T011 : customers-list, customers-new
T012 : customers-list, customers-search, customers-detail
T013 : customers-list, customers-search, customers-detail, customers-edit
T014 : orders-list, orders-new
T015 : orders-list, orders-detail, orders-updated
T016 : orders-list, orders-detail, orders-updated
T017 : orders-list, orders-detail, orders-updated
T018 : orders-list, orders-filtered
T019 : tickets-list, tickets-detail, tickets-assigned


## Việc còn lại
Trung xác nhận hợp đồng v0.1 và nối candidate generator; hai thành viên bổ sung gán nhãn phần tử đích và đo recall/số ứng viên. Bộ mẫu này chưa phủ mọi bước 20 task, chưa có iframe, shadow DOM hoặc mutation. Không suy ra độ bền locator từ kiểm tra XPath ở DOM hiện tại.
